In [ ]:
"""
Quando selecionamos uma imagem em Markdown num Jupyter Notebook, usamos um caminho relativo ao caminho do Notebook
"""

In [1]:

from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
import librosa

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

import json
import jiwer #https://github.com/jitsi/jiwer
import time

import numpy as np

c:\Users\Admin\Desktop\ip\Automatic Speech Recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<h4> Data Loader </h4>

In [2]:
with open (r"C:\Users\Admin\Desktop\ip\Automatic Speech Recognition\v2\Eval\audios.json", "r", encoding = "utf-8") as f:
    json_file = json.load (f)

display (json_file)

"""
    Normalização do Dataset.
    É importante que tanto o dataset como a transcrição obtida pelo modelo ASR tenham a mesma formatação por isso ambos vão passar por um normalização simples de remoção de duplos espaços e conversão 
    de todas as palavras para lower case.
"""

dataset = []

for exemplo in json_file:
    """
        JOIN reconstrói a frase deixando a mesma apenas com espaços em branco normais. 
        SPLIT ajuda o JOIN, separando todas as palavras da frase.
        LOWER é auto explicativo.
    """
    dataset.append (" ".join(str(exemplo["trans"]).split()).lower())

display (dataset)

[{'audio_id': 1,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio1.wav',
  'trans': 'Boa tarde É aí da papelaria Sim O menina dá para me guardar dois bilhetes Dois bilhetes  Sim Que bilhetes diga-me Bilhetes lá do coiso que vai acontecer na Sexta Qual é o espetáculo diga-me É lá o que acontece lá no casino que o meu neto é que quer ir Mas eu não sei qual é Você sabe  É o da Sexta sei lá o meu neto é que me pediu isto já liguei para aqui para tantos sitíos tenho que lá ir tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam É assim mas você para comprar tem que vir cá pagá-los Sim está bem mas tem que mos guardar não é  Não tem de vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora não posso guardá-los Então e depois eu chego aí ',
  'duration': 42},
 {'audio_id': 2,
  'audio_path': 'C:/Users/Admin/Desktop/ip/Automatic Speech Recognition/audio/audio2.wav',
  'trans': 'Boa tarde O m

['boa tarde é aí da papelaria sim o menina dá para me guardar dois bilhetes dois bilhetes sim que bilhetes diga-me bilhetes lá do coiso que vai acontecer na sexta qual é o espetáculo diga-me é lá o que acontece lá no casino que o meu neto é que quer ir mas eu não sei qual é você sabe é o da sexta sei lá o meu neto é que me pediu isto já liguei para aqui para tantos sitíos tenho que lá ir tenho que lá ir e agora disseram-me porque é que não liga lá pá papelaria que eles guardam é assim mas você para comprar tem que vir cá pagá-los sim está bem mas tem que mos guardar não é não tem de vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora não posso guardá-los então e depois eu chego aí',
 'boa tarde o menina até estou nervosa que vim de lá de baixo o menina eu já encontrei encontrou o quê encontrei a elisa a então eu vou passar aqui ás relações públicas e diz está bem só um momento relações públicas boa tarde olhe o menino eu já encontrei encontrou o quê a 

<h5> Model Loader </h5>

In [4]:
MODEL_PATH = r"C:\Users\Admin\Desktop\models\ASR Models\Whisper\Whisper-large-v3 4Bit"

PROCESSOR = AutoProcessor.from_pretrained (MODEL_PATH)
MODEL = AutoModelForSpeechSeq2Seq.from_pretrained (MODEL_PATH, device_map = device, dtype = torch.float16)

W0908 14:57:00.789000 13808 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 1259/1259 [00:02<00:00, 596.63it/s]


<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 1 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V1.png">
</p>

In [5]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt")

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

In [6]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[14.563031435012817, 13.748192310333252, 10.188478469848633, 17.229735136032104, 13.331352710723877, 15.173702478408813, 11.044028997421265, 9.607661724090576, 10.422703504562378, 11.060672760009766, 20.938469409942627, 21.66313076019287, 16.672937631607056, 20.324735164642334]


['e foi, boa tarde ei, da papelaria sim oh menina, dá para me guardar dois bilhetes? dois bilhetes? sim que bilhetes, diga-me bilhetes lá do coisa que vai acontecer na sexta qual é o espetáculo, diga-me é logo o que acontece lá no casino com o meu neto, querido mas eu não sei qual é, você sabe? é o da sexta, sei lá, o meu neto é com o padrinho já liguei para aqui, para cá, para o sítio tem que ir lá, tem que ir lá e agora disseram, porquê que não liga lá para a papelaria que eles guardam? é sim, mas você para comprar tem que ficar para graus. sim, está bem, mas tem que nos guardar, não é? não, tem que ficar para eu tirar na máquina, sair da impressora o papel, para você me pagar na hora, não posso guardar os. então, e depois eu cheguei...',
 'tv, boa tarde. oh, menina. estou nervosa, cabelo de lado baixo. oh, menina, eu já encontrei. encontrou o quê? encontrei a elisa. ah, então eu vou passar aqui às relações públicas e diz, está bem? olá, elisa. relações públicas, boa tarde. olha, men

{'WER_CALC': [0.41830065359477125, 0.4144144144144144, 0.3898305084745763, 0.4019607843137255, 0.36538461538461536, 0.22377622377622378, 0.4117647058823529, 0.6439024390243903, 0.39361702127659576, 0.38095238095238093, 0.325, 0.42342342342342343, 0.40119760479041916, 0.3271889400921659], 1: 0.41830065359477125, 2: 0.4144144144144144, 3: 0.3898305084745763, 4: 0.4019607843137255, 5: 0.36538461538461536, 6: 0.22377622377622378, 7: 0.4117647058823529, 8: 0.6439024390243903, 9: 0.39361702127659576, 10: 0.38095238095238093, 11: 0.325, 12: 0.42342342342342343, 13: 0.40119760479041916, 14: 0.3271889400921659}
{'CER_CALC': [0.18169209431345354, 0.15300546448087432, 0.18197573656845753, 0.17185929648241205, 0.21015514809590974, 0.11512717536813923, 0.2080808080808081, 0.5351681957186545, 0.1740139211136891, 0.15046296296296297, 0.09025270758122744, 0.18412162162162163, 0.2068527918781726, 0.17290552584670232], 1: 0.18169209431345354, 2: 0.15300546448087432, 3: 0.18197573656845753, 4: 0.17185929

<p align = "center">
    <img src = "evals\Latex Table Sistema V1.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 2 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V2.png">
</p>

In [7]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5)

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [8]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[17.68185567855835, 13.70716667175293, 12.667884111404419, 22.042357206344604, 15.754745721817017, 16.454543828964233, 13.037170171737671, 11.503896236419678, 10.480395078659058, 9.455599546432495, 18.964371919631958, 25.4699649810791, 18.55312943458557, 22.94831395149231]


['e foi, boa tarde ei, da papelaria sim oh, menina, dá para me guardar dois bilhetes? dois bilhetes? sim que bilhetes, diga-me bilhetes lá do coisa que vai acontecer na sexta qual é o espetáculo, diga-me é logo o que acontece lá no casino com o meu neto, querido mas eu não sei qual é, você sabe? é o da sexta, sei lá, o meu neto é compadre eu já liguei para aqui, portanto, o sítio tem que lá ir, tem que lá ir e agora disseram, porquê que não liga lá para a papelaria que eles guardam? é sim, mas você para comprar tem que vir cá pagar os sim, está bem, mas tem que nos guardar, não é? não, tem que vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora, não posso guardar os então, e depois eu cheguei',
 'tv, boa tarde. oh, menina, estou nervosa que vim do lado baixo. oh, menina, eu já encontrei. encontrou o quê? encontrei a elisa. ah, então eu vou passar aqui às relações públicas e diz, está bem? olá, elisa. relações públicas, boa tarde. olha, menina, eu já en

{'WER_CALC': [0.35947712418300654, 0.3783783783783784, 0.3898305084745763, 0.3235294117647059, 0.391025641025641, 0.21678321678321677, 0.4117647058823529, 0.6878048780487804, 0.3829787234042553, 0.2261904761904762, 0.28125, 0.42342342342342343, 0.33532934131736525, 0.2857142857142857], 1: 0.35947712418300654, 2: 0.3783783783783784, 3: 0.3898305084745763, 4: 0.3235294117647059, 5: 0.391025641025641, 6: 0.21678321678321677, 7: 0.4117647058823529, 8: 0.6878048780487804, 9: 0.3829787234042553, 10: 0.2261904761904762, 11: 0.28125, 12: 0.42342342342342343, 13: 0.33532934131736525, 14: 0.2857142857142857}
{'CER_CALC': [0.14285714285714285, 0.1366120218579235, 0.18197573656845753, 0.15175879396984926, 0.2736248236953456, 0.11378848728246319, 0.21414141414141413, 0.5443425076452599, 0.16705336426914152, 0.125, 0.10108303249097472, 0.18496621621621623, 0.14593908629441624, 0.1497326203208556], 1: 0.14285714285714285, 2: 0.1366120218579235, 3: 0.18197573656845753, 4: 0.15175879396984926, 5: 0.273

<p align = "center">
    <img src = "evals\Latex Table Sistema V2.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 3 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V3.png">
</p>

In [9]:
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)

    """
        ###############################
    """
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False)

    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 10)

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)

    """
        ###############################
    """

    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [10]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[21.3340060710907, 18.022650718688965, 16.886212825775146, 30.647709846496582, 23.000272035598755, 23.660123348236084, 15.678457975387573, 13.24058723449707, 13.159897565841675, 12.014532566070557, 25.69551706314087, 35.19501757621765, 20.75317120552063, 27.95683240890503]


['e foi, boa tarde ei, da papelaria sim oh, menina, dá para me guardar dois bilhetes? dois bilhetes? sim que bilhetes, diga-me bilhetes lá do coisa que vai acontecer na sexta qual é o espetáculo, diga-me é logo o que acontece lá no casino com o meu neto, querido mas eu não sei qual é, você sabe? é o da sexta, sei lá, o meu neto é compadre eu já liguei para aqui, portanto, o sítio tem que lá ir, tem que lá ir e agora disseram, porquê que não liga lá para a papelaria que eles guardam? é sim, mas você para comprar tem que vir cá pagar-os. sim, está bem, mas tem que nos guardar, não é? não, tem que vir cá para eu tirar na máquina, sair da impressora o papel, para você me pagar na hora, não posso guardá-los. então, e depois eu cheguei...',
 'tv, boa tarde. oh, menina, estou nervosa que vim do lado baixo. oh, menina, eu já encontrei. encontrou o quê? encontrei a elisa. ah, então eu vou passar aqui às relações públicas e diz, está bem? olá, elisa. relações públicas, boa tarde. olha, menina, e

{'WER_CALC': [0.35947712418300654, 0.3963963963963964, 0.3898305084745763, 0.3284313725490196, 0.41025641025641024, 0.21678321678321677, 0.4117647058823529, 0.6780487804878049, 0.3829787234042553, 0.2261904761904762, 0.2625, 0.42342342342342343, 0.2754491017964072, 0.2488479262672811], 1: 0.35947712418300654, 2: 0.3963963963963964, 3: 0.3898305084745763, 4: 0.3284313725490196, 5: 0.41025641025641024, 6: 0.21678321678321677, 7: 0.4117647058823529, 8: 0.6780487804878049, 9: 0.3829787234042553, 10: 0.2261904761904762, 11: 0.2625, 12: 0.42342342342342343, 13: 0.2754491017964072, 14: 0.2488479262672811}
{'CER_CALC': [0.14701803051317613, 0.14025500910746813, 0.18197573656845753, 0.15477386934673368, 0.2778561354019746, 0.11378848728246319, 0.21414141414141413, 0.5514780835881753, 0.16705336426914152, 0.125, 0.08182912154031288, 0.18412162162162163, 0.1281725888324873, 0.14171122994652408], 1: 0.14701803051317613, 2: 0.14025500910746813, 3: 0.18197573656845753, 4: 0.15477386934673368, 5: 0.2

<p align = "center">
    <img src = "evals\Latex Table Sistema V3.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 4 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V4.png">
</p>

In [11]:
from silero_vad import load_silero_vad, get_speech_timestamps

VAD_MODEL = load_silero_vad ()

In [16]:
"""
Este código calcula WER, CER, RTF e Latência do sistema Assobio com Pré Processamento de Áudio com Voice Activity Detection (VAD).
Este pedaço de código foi rodado nas 3 seguintes configurações:

    Greedy Search
    Beam Decoding = 5
    Beam Decoding = 10

Como VAD transforma um áudio em vários áudios (excertos), temos que iterar sobre os mesmos. A transcrição de um áudio é o fim do loop dos excertos do VAD e este código consegue realizar isso criando a lista ADD a 
cada áudio novo.
Também surge a questão da Latência porque a Latência corresponde ao tempo de transcrição do áudio inteiro e não apenas de um excerto, para isso usei uma variável para ir adicionando o tempo que demorava em cada excerto.
total_lat_ex = 0

Os resultados estão no MD seguinte.
"""

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)
    VOZ = get_speech_timestamps (WAV, VAD_MODEL)

    ADD = []
    total_lat_ex = 0

    for excertos in VOZ:

        WAV_VOZ = WAV [excertos["start"] : excertos ["end"]]

        torch.cuda.synchronize ()
        begin = time.time ()
    
        inputs = PROCESSOR (WAV_VOZ, sampling_rate = 16000, return_tensors = "pt", truncation = False) 
        inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

        with torch.inference_mode ():
            outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 10) # num_beams = 5

        torch.cuda.synchronize ()
        time_lat = time.time () - begin
        total_lat_ex += time_lat

        trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
        trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

        ADD.append (trans)

    LATENCY.append (total_lat_ex)

    full_trans = " ".join (ADD)

    MODEL_TRANS.append (full_trans)

    wer = jiwer.wer (dataset[idx], full_trans)
    cer = jiwer.cer (dataset[idx], full_trans)
    rtf = total_lat_ex / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [17]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[21.511265993118286, 20.032138109207153, 20.794602155685425, 29.236801385879517, 25.498282194137573, 27.06229066848755, 19.311115741729736, 33.630786657333374, 14.76021122932434, 14.21817660331726, 23.141867637634277, 34.92732000350952, 28.510353088378906, 35.36472797393799]


['boa tarde. é aí da papelaria? sim. oh menina, dá para me guardar dois bilhetes? sim. bilhete do lado ou do coiso que vai acontecer na sexta. é logo o que acontece lá no casino com a minha neta querida. você sabe? eu descei, descei lá, o meu neto me pediu, já liguei para aqui, portanto, o sítio tem que lá ir, tem que lá ir. e agora disseram, por que não liga lá para a papelaria que eles guardam? mas você, para comprar, tem que vir cá pagar-os. sim, está bem, mas tem que nos guardar, não é? não, tem que vir cá para eu tirar na máquina, sair da impressora ao papel, para você me pagar na hora, não posso guardá-los. então, e depois eu cheguei...',
 'boa tarde amanina. é toda nervosa que vem de lá de baixo homem, eu já encontrei contra o quê? encontrei lisa. então eu vou passar aqui às relações públicas e diz, está bem? rolações públicas, boa tarde olha, menina, eu já encontrei. encontrou o quê? a elisa, que ela mora aqui embaixo. ah, ok. vocês todas as noites perguntam. uma vizinha minha 

{'WER_CALC': [0.5228758169934641, 0.38738738738738737, 0.423728813559322, 0.4117647058823529, 0.532051282051282, 0.26573426573426573, 0.5490196078431373, 0.5365853658536586, 0.425531914893617, 0.5238095238095238, 0.45, 0.4369369369369369, 0.3772455089820359, 0.3317972350230415], 1: 0.5228758169934641, 2: 0.38738738738738737, 3: 0.423728813559322, 4: 0.4117647058823529, 5: 0.532051282051282, 6: 0.26573426573426573, 7: 0.5490196078431373, 8: 0.5365853658536586, 9: 0.425531914893617, 10: 0.5238095238095238, 11: 0.45, 12: 0.4369369369369369, 13: 0.3772455089820359, 14: 0.3317972350230415}
{'CER_CALC': [0.26907073509015256, 0.15300546448087432, 0.17504332755632582, 0.2271356783919598, 0.38081805359661497, 0.1111111111111111, 0.30505050505050507, 0.28338430173292556, 0.2111368909512761, 0.37037037037037035, 0.2250300842358604, 0.20777027027027026, 0.19035532994923857, 0.15597147950089127], 1: 0.26907073509015256, 2: 0.15300546448087432, 3: 0.17504332755632582, 4: 0.2271356783919598, 5: 0.380

<p align = "center">
    <img src = "evals\Latex Table Sistema V4.1.png">
</p>

<p align = "center">
    <img src = "evals\Latex Table Sistema V4.2.png">
</p>

<p align = "center">
    <img src = "evals\Latex Table Sistema V4.3.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 5 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V5.png">
</p>

In [18]:
from demucs.api import Separator
import torchaudio

SEPARATOR_DEMUCS = Separator (model = "htdemucs")

In [19]:
"""
Nesta versão usamos a biblioteca Demucs da Meta para podermos separar o Ruído das Vozes.
A biblioteca recebe o áudio a 44.1kHz e recebe logo um tensor PyTorch por isso temos que dar load do áudio nesse sample rate e convertido para PyTorch.
Após isso temos que converter o áudio para 16khZ e em Mono por isso usamos torchaudio para converter para 16kHz e usamos a média para converter de 2 canais para 1 canal.
"""
LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 44100, mono = False) #44.1kHz | Estéreo
    WAV = torch.from_numpy (WAV)
    
    ORIGINAL, SEPARADO = SEPARATOR_DEMUCS.separate_tensor (WAV, sr = SAMPLING_RATE)

    VOZES = SEPARADO ["vocals"]
    VOZES = torchaudio.functional.resample (VOZES, orig_freq = SAMPLING_RATE, new_freq = 16000)
    VOZES = VOZES.mean (dim = 0)

    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (VOZES, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [20]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[18.182165145874023, 14.887761354446411, 13.51197338104248, 25.229763984680176, 17.072940587997437, 16.823853015899658, 13.562453508377075, 11.93622088432312, 11.275615215301514, 9.965378284454346, 18.526257276535034, 47.849265336990356, 22.029852628707886, 27.89781641960144]


['e foi, boa tarde ei, da papelaria sim oh, menina, dá para me guardar dois bilhetes? dois bilhetes? sim que bilhetes, diga-me bilhetes lá do coisa que vai acontecer na sexta qual é o espetáculo, diga-me é logo o que acontece lá no casino com o meu neto, querido mas eu não sei qual é, você sabe? é o da sexta, sei lá, o meu neto é compadre eu já liguei para aqui, portanto, o sítio tem que lá ir, tem que lá ir e agora disseram, porquê que não liga lá para a papelaria que eles guardam? é sim, mas você para comprar tem que vir cá pagar os sim, está bem, mas tem que nos guardar, não é? não, tem que vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora, não posso guardar os então, e depois eu cheguei',
 'tv, boa tarde. oh, menina, estou nervosa que vim do lado baixo. oh, menina, eu já encontrei. encontrou o quê? encontrei a elisa. ah, então eu vou passar aqui às voçãs públicas e diz, está bem? olá, voçãs públicas, boa tarde. olha, menina, eu já encontrei. enco

{'WER_CALC': [0.35947712418300654, 0.4144144144144144, 0.3898305084745763, 0.3333333333333333, 0.391025641025641, 0.21678321678321677, 0.4117647058823529, 0.6780487804878049, 0.3829787234042553, 0.2261904761904762, 0.19375, 1.1306306306306306, 0.32335329341317365, 0.31336405529953915], 1: 0.35947712418300654, 2: 0.4144144144144144, 3: 0.3898305084745763, 4: 0.3333333333333333, 5: 0.391025641025641, 6: 0.21678321678321677, 7: 0.4117647058823529, 8: 0.6780487804878049, 9: 0.3829787234042553, 10: 0.2261904761904762, 11: 0.19375, 12: 1.1306306306306306, 13: 0.32335329341317365, 14: 0.31336405529953915}
{'CER_CALC': [0.14285714285714285, 0.16029143897996356, 0.18197573656845753, 0.15678391959798996, 0.28067700987306066, 0.11378848728246319, 0.21414141414141413, 0.5412844036697247, 0.16473317865429235, 0.125, 0.07701564380264742, 0.6182432432432432, 0.14086294416243655, 0.21122994652406418], 1: 0.14285714285714285, 2: 0.16029143897996356, 3: 0.18197573656845753, 4: 0.15678391959798996, 5: 0.

<p align = "center">
    <img src = "evals\Latex Table Sistema V5.png">
</p>

<hr>

<h3> Avaliação do Sistema <b> Assobio</b> Versão 6 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V6.png">
</p>


In [21]:
import noisereduce as nr #https://github.com/timsainb/noisereduce

In [22]:

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True) 
    WAV = torch.from_numpy (WAV)
    
    WAV = nr.reduce_noise (WAV, sr = SAMPLING_RATE)

    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [23]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[18.882818698883057, 16.725383520126343, 16.01175880432129, 25.35254144668579, 18.720475912094116, 19.9260892868042, 15.529712438583374, 11.395122528076172, 12.460893392562866, 13.63199257850647, 22.16471004486084, 21.92045569419861, 19.80572509765625, 68.46397686004639]


['e foi, boa tarde. é aí da papelaria? sim. oh, menina, dá para me guardar dois bilhetes? dois bilhetes? sim. que bilhetes, diga-me? bilhetes lá do coisa que vai acontecer na sexta. qual é o espetáculo, diga-me? o que acontece lá no casino, como eu não tenho que dizer. mas eu não sei qual é, você sabe? é da sexta, sei lá, o meu neto, como pediu, já liguei para mim. eu sinto que tenho que ir lá, tenho que ir lá. e agora disseram, porquê que não liga lá para a papelaria, que eles guardam. é sim, mas você para comprar tem que vir cá pagar os. sim, está bem, mas tem que nos guardar, não é? não, sim, vir cá, põe o eixo a tirar na máquina e sai de impressora ao papel para você me pagar na hora, não posso guardá-los. então, e depois eu cheguei...',
 'tive boa tarde. oh menina, estou nervosa que vim de lá de baixo. oh menina, eu já encontrei. encontrou o quê? encontrei a elisa. ah, então eu vou passar aqui as vozes públicas. está bem, está bem. estava a fazer as vozes públicas, boa tarde. olha

{'WER_CALC': [0.5032679738562091, 0.43243243243243246, 0.5423728813559322, 0.4411764705882353, 0.48717948717948717, 0.26573426573426573, 0.6470588235294118, 0.6731707317073171, 0.40425531914893614, 0.36904761904761907, 0.3625, 0.5675675675675675, 0.41317365269461076, 1.2672811059907834], 1: 0.5032679738562091, 2: 0.43243243243243246, 3: 0.5423728813559322, 4: 0.4411764705882353, 5: 0.48717948717948717, 6: 0.26573426573426573, 7: 0.6470588235294118, 8: 0.6731707317073171, 9: 0.40425531914893614, 10: 0.36904761904761907, 11: 0.3625, 12: 0.5675675675675675, 13: 0.41317365269461076, 14: 1.2672811059907834}
{'CER_CALC': [0.19972260748959778, 0.17122040072859745, 0.2582322357019064, 0.22110552763819097, 0.33568406205923834, 0.14190093708165996, 0.34545454545454546, 0.5688073394495413, 0.17633410672853828, 0.16435185185185186, 0.1263537906137184, 0.41047297297297297, 0.2017766497461929, 0.9901960784313726], 1: 0.19972260748959778, 2: 0.17122040072859745, 3: 0.2582322357019064, 4: 0.2211055276

<p align = "center">
    <img src = "evals\Latex Table Sistema V6.png">
</p>

<hr>
<h3> Avaliação do Sistema <b> Assobio</b> Versão 7 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V7.png">
</p>

In [24]:

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True) 
    WAV = torch.from_numpy (WAV)
    
    WAV = WAV / torch.max (torch.abs (WAV))

    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (WAV, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


In [25]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[21.07171940803528, 17.299941539764404, 15.773663997650146, 28.125329971313477, 19.12639045715332, 21.035274267196655, 15.610299348831177, 15.426251649856567, 12.26167631149292, 13.627235174179077, 22.725244283676147, 26.924095630645752, 17.622185468673706, 23.017489433288574]


['e foi, boa tarde ei, da papelaria sim oh, menina, dá para me guardar dois bilhetes? dois bilhetes? sim que bilhetes, diga-me bilhetes lá do coisa que vai acontecer na sexta qual é o espetáculo, diga-me é logo o que acontece lá no casino com o meu neto, querido mas eu não sei qual é, você sabe? é o da sexta, sei lá, o meu neto é compadre eu já liguei para aqui, portanto, o sítio tem que lá ir, tem que lá ir e agora disseram, porquê que não liga lá para a papelaria que eles guardam? é sim, mas você para comprar tem que vir cá pagar os sim, está bem, mas tem que nos guardar, não é? não, tem que vir cá para eu tirar na máquina sair da impressora o papel para você me pagar na hora, não posso guardar os então, e depois eu cheguei',
 'tv, boa tarde. oh, menina, estou nervosa que vim do lado baixo. oh, menina, eu já encontrei. encontrou o quê? encontrei a elisa. ah, então eu vou passar aqui às relações públicas e diz, está bem? olá, elisa. relações públicas, boa tarde. olha, menina, eu já en

{'WER_CALC': [0.35947712418300654, 0.3963963963963964, 0.3898305084745763, 0.3284313725490196, 0.3974358974358974, 0.21678321678321677, 0.4117647058823529, 0.6878048780487804, 0.3829787234042553, 0.36904761904761907, 0.26875, 0.4099099099099099, 0.281437125748503, 0.2857142857142857], 1: 0.35947712418300654, 2: 0.3963963963963964, 3: 0.3898305084745763, 4: 0.3284313725490196, 5: 0.3974358974358974, 6: 0.21678321678321677, 7: 0.4117647058823529, 8: 0.6878048780487804, 9: 0.3829787234042553, 10: 0.36904761904761907, 11: 0.26875, 12: 0.4099099099099099, 13: 0.281437125748503, 14: 0.2857142857142857}
{'CER_CALC': [0.14285714285714285, 0.14025500910746813, 0.18197573656845753, 0.1527638190954774, 0.27926657263751764, 0.11378848728246319, 0.21414141414141413, 0.5443425076452599, 0.16705336426914152, 0.14583333333333334, 0.09025270758122744, 0.17567567567567569, 0.12944162436548223, 0.1497326203208556], 1: 0.14285714285714285, 2: 0.14025500910746813, 3: 0.18197573656845753, 4: 0.1527638190954

<p align = "center">
    <img src = "evals\Latex Table Sistema V7.png">
</p>

<hr>
<h3> Avaliação do Sistema <b> Assobio</b> Versão 8 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V8.png">
</p>

In [26]:
from demucs.api import Separator
import torchaudio

DEMUCS_SEPARATOR = Separator (model = "htdemucs")

In [27]:

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 44100, mono = False) 
    WAV = torch.from_numpy (WAV)

    if len (WAV.shape) == 1:
        WAV = WAV.unsqueeze (0) #https://stackoverflow.com/questions/57237352/what-does-unsqueeze-do-in-pytorch
        WAV = WAV.repeat (2, 1) #https://docs.pytorch.org/docs/2.14/generated/torch.Tensor.repeat.html

    WAV = WAV / torch.max (torch.abs (WAV))

    ORIGINAL, SEPARADO = DEMUCS_SEPARATOR.separate_tensor (WAV, sr = SAMPLING_RATE)

    VOZES = SEPARADO ["vocals"]
    VOZES = torchaudio.functional.resample (VOZES, orig_freq = SAMPLING_RATE, new_freq = 16000)
    VOZES = VOZES.mean (dim = 0)

    VOZES = VOZES / torch.max (torch.abs (VOZES))

    GANHO_DB = 6
    GANHO = 10 ** (GANHO_DB / 20)
    VOZES = VOZES * GANHO

    print (VOZES)
    print (VOZES.shape)
    torch.cuda.synchronize ()
    begin = time.time ()
    
    inputs = PROCESSOR (VOZES, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
    inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

    with torch.inference_mode ():
        outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt", num_beams = 5) # num_beams = 5

    torch.cuda.synchronize ()
    time_lat = time.time () - begin
    LATENCY.append (time_lat)
    
    trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
    trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

    MODEL_TRANS.append (trans)

    wer = jiwer.wer (dataset[idx], trans)
    cer = jiwer.cer (dataset[idx], trans)
    rtf = time_lat / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


tensor([ 0.0182, -0.0089, -0.0319,  ...,  0.3913, -0.4740, -0.3804])
torch.Size([672000])
tensor([-0.8701, -1.3379, -0.4624,  ..., -0.4589, -0.3577, -0.2230])
torch.Size([520381])
tensor([-0.0279,  0.4445,  0.7249,  ..., -0.1203, -0.3812, -0.2681])
torch.Size([511381])
tensor([ 0.6022,  1.2610,  0.8203,  ..., -0.0470, -0.0350, -0.0063])
torch.Size([869792])
tensor([ 0.0017,  0.0056,  0.0080,  ..., -0.0554, -0.0563, -0.0180])
torch.Size([801056])
tensor([ 0.0012,  0.0099,  0.0234,  ..., -0.0126, -0.0128, -0.0010])
torch.Size([766845])
tensor([ 0.0603,  0.0105, -0.0723,  ...,  0.0075, -0.0526, -0.0184])
torch.Size([465023])
tensor([ 0.0064, -0.0117, -0.0380,  ...,  0.0960,  0.6724,  0.7240])
torch.Size([866836])
tensor([ 1.5762e-04,  8.1070e-04,  1.4701e-03,  ...,  4.4403e-01,
         2.7323e-01, -1.5428e-01])
torch.Size([416102])
tensor([ 0.4677,  0.8179,  0.7397,  ...,  0.0025, -0.8338, -1.0437])
torch.Size([473165])
tensor([ 0.0706,  0.0318,  0.0247,  ...,  0.0148, -0.0188, -0.0309])

In [28]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[19.714771509170532, 13.488404512405396, 12.864272594451904, 22.345953226089478, 16.96865224838257, 17.17949891090393, 13.963615655899048, 12.01609182357788, 11.301027536392212, 10.343425750732422, 18.005220890045166, 21.64495301246643, 16.840582847595215, 23.206351041793823]


['o que foi? boa tarde. ei, da papelaria. sim? oh, menina, dá para me guardar dois bilhetes? dois bilhetes? sim. que bilhetes, diga-me? bilhetes lá do coisa que vai acontecer na sexta. qual é o espetáculo, diga-me? é logo o que acontece lá no casino com o meu neto, querido. mas eu não sei qual é. você sabe? é o da sexta, sei lá, o meu neto é com o pedido, já liguei para cá, para cá, do sítio, tem que lá ir, tem que lá ir. e agora disseram, porquê que não liga lá para a papelaria que eles guardam? sim, está bem, mas temos que nos guardar, não é? não. tem que ficar para eu tirar na máquina, sair da impressora o papel, para você me pagar',
 'tv, boa tarde oh menina, estou nervosa que vim do lado baixo oh menina, eu já encontrei encontrou o quê? encontrei a elisa ah, então eu vou passar aqui às voçãs públicas e diz, está bem? olá, voçãs públicas, boa tarde olha, menina, eu já encontrei encontrou o quê? a elisa, que ela mora aqui embaixo ah, ok vocês todas as noites perguntam uma vizinha mi

{'WER_CALC': [0.5359477124183006, 0.32432432432432434, 0.3983050847457627, 0.3235294117647059, 0.4230769230769231, 0.21678321678321677, 0.4117647058823529, 0.6878048780487804, 0.39361702127659576, 0.38095238095238093, 0.1875, 0.4954954954954955, 0.3532934131736527, 0.29493087557603687], 1: 0.5359477124183006, 2: 0.32432432432432434, 3: 0.3983050847457627, 4: 0.3235294117647059, 5: 0.4230769230769231, 6: 0.21678321678321677, 7: 0.4117647058823529, 8: 0.6878048780487804, 9: 0.39361702127659576, 10: 0.38095238095238093, 11: 0.1875, 12: 0.4954954954954955, 13: 0.3532934131736527, 14: 0.29493087557603687}
{'CER_CALC': [0.30374479889042993, 0.1366120218579235, 0.18370883882149047, 0.1507537688442211, 0.28772919605077574, 0.11780455153949129, 0.21414141414141413, 0.5565749235474006, 0.18329466357308585, 0.15046296296296297, 0.06738868832731648, 0.2660472972972973, 0.1548223350253807, 0.18627450980392157], 1: 0.30374479889042993, 2: 0.1366120218579235, 3: 0.18370883882149047, 4: 0.150753768844

<p align = "center">
    <img src = "evals\Latex Table Sistema V8.png">
</p>

<hr>
<h3> Avaliação do Sistema <b> Assobio</b> Versão 9 </h3>

<p align = "center">
    <img src = "system diagram\Sistema Assobio V9.png">
</p>

In [29]:
from demucs.api import Separator
import torchaudio
from silero_vad import load_silero_vad, get_speech_timestamps

DEMUCS_SEPARATOR = Separator (model = "htdemucs")
VAD_MODEL = load_silero_vad ()

In [30]:
"""

"""

LATENCY = []

MODEL_TRANS = []
WER = {
    "WER_CALC": []
}
CER = {
    "CER_CALC": []
}
RTF = {
    "RTF_CALC": []
}


for idx, line in enumerate (json_file):
    #print (line["audio_path"])
    WAV, SAMPLING_RATE = librosa.load (line["audio_path"], sr = 16000, mono = True)
    #print (WAV)
    VOZ = get_speech_timestamps (WAV, VAD_MODEL)
    print (VOZ)

    ADD = []
    total_lat_ex = 0

    for excertos in VOZ:

        WAV2 = WAV [excertos["start"] : excertos ["end"]]

        WAV2 = torch.from_numpy (WAV2)
        WAV2 = WAV2.unsqueeze (0)
        WAV2 = WAV2.repeat (2, 1)
        WAV2 = torchaudio.functional.resample (WAV2, orig_freq = 16000, new_freq = 44100)

        ORIGINAL, SEPARADO = DEMUCS_SEPARATOR.separate_tensor (WAV2, sr = 44100)
        
        VOZES = SEPARADO ["vocals"]
        
        VOZES = torchaudio.functional.resample (VOZES, orig_freq = 44100, new_freq = 16000)
                
        VOZES = VOZES / torch.max (torch.abs (VOZES))
        VOZES = VOZES.mean (dim = 0)

        GANHO = 10 ** (6 / 20)
        VOZES = VOZES * GANHO

        #print (VOZES)
        torch.cuda.synchronize ()
        begin = time.time ()
    
        inputs = PROCESSOR (VOZES, sampling_rate = 16000, return_tensors = "pt", truncation = False) #, truncation = False
        inputs = inputs["input_features"].to (device, dtype = torch.float16) # Passar para GPU, não faz parte do Prefill

        with torch.inference_mode ():
            outputs = MODEL.generate (inputs, return_timestamps = True, task = "transcribe", language = "pt") # num_beams = 5

        torch.cuda.synchronize ()
        time_lat = time.time () - begin
        total_lat_ex += time_lat

        trans = PROCESSOR.batch_decode (outputs, skip_special_tokens = True)[0]
        trans = " ".join(str(trans).split()).lower() #Normalização tal como no dataset original

        ADD.append (trans)

    LATENCY.append (total_lat_ex)

    full_trans = " ".join (ADD)

    MODEL_TRANS.append (full_trans)

    wer = jiwer.wer (dataset[idx], full_trans)
    cer = jiwer.cer (dataset[idx], full_trans)
    rtf = total_lat_ex / line["duration"]

    WER[line["audio_id"]] = wer
    WER["WER_CALC"].append (wer)

    CER[line["audio_id"]] = cer
    CER["CER_CALC"].append (cer)

    RTF[line["audio_id"]] = rtf
    RTF["RTF_CALC"].append (rtf)


[{'start': 37920, 'end': 69600}, {'start': 73760, 'end': 116704}, {'start': 133152, 'end': 143328}, {'start': 161312, 'end': 211936}, {'start': 228896, 'end': 300000}, {'start': 305696, 'end': 311264}, {'start': 314912, 'end': 456672}, {'start': 462368, 'end': 672000}]
[{'start': 9760, 'end': 17376}, {'start': 23072, 'end': 35296}, {'start': 42528, 'end': 102880}, {'start': 107040, 'end': 116704}, {'start': 118816, 'end': 143840}, {'start': 157216, 'end': 246752}, {'start': 249888, 'end': 520381}]
[{'start': 2592, 'end': 9184}, {'start': 12832, 'end': 100320}, {'start': 102432, 'end': 200160}, {'start': 205344, 'end': 234976}, {'start': 236576, 'end': 511380}]
[{'start': 30752, 'end': 186336}, {'start': 187936, 'end': 220128}, {'start': 249376, 'end': 380384}, {'start': 383008, 'end': 652256}, {'start': 653856, 'end': 869791}]
[{'start': 49696, 'end': 55264}, {'start': 65568, 'end': 137696}, {'start': 139296, 'end': 320992}, {'start': 334368, 'end': 346080}, {'start': 348704, 'end': 38

In [31]:
"""
Resultados
"""

print (LATENCY)
display (MODEL_TRANS)
print (WER)
print (CER)
print (RTF) #Real Time Factor -> 

print ("---" *50)

print (f"Média da Latência: {np.mean (LATENCY)} segundos")
print (f"P50 Latência: {np.percentile (LATENCY, 50)} segundos")
print (f"P95 Latência: {np.percentile (LATENCY, 95)} segundos")

print ("---" * 50)

print (f"Média Word Error Rate: {np.mean(WER['WER_CALC'])}")
print (f"Média Character Error Rate: {np.mean(CER['CER_CALC'])}")
print (f"Média Real Time Factor: {np.mean(RTF['RTF_CALC'])}")
print (f"P50 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 50)}")
print (f"P95 Real Time Factor: {np.percentile(RTF['RTF_CALC'], 95)}")

[15.118749380111694, 13.987490177154541, 14.615036010742188, 20.165652990341187, 15.760494947433472, 16.355270385742188, 14.06311297416687, 20.08528757095337, 8.343488931655884, 6.5945892333984375, 13.584383964538574, 18.301316738128662, 14.910048961639404, 23.711304664611816]


['é aí da papelaria. oh menina, dá para me guardar dois bilhetes? sim. bilhete de lado ou do coiso que vai acontecer na sexta. é logo o que acontece lá no casino com o meu neto aqui a rir. você sabe? eu descei, descei lá, o meu neto é com o pedido, já liguei para aqui, para cá, para o sítio, tenho que ir, tenho que ir. e agora disseram, por que não liga lá para a papelaria que eles guardam? mas você para comprar tem que ficar para lá. sim, está bem, mas tem que me guardar, não é? não. tem que ficar para eu tirar na máquina, sair da impressora o papel, para você me pagar na hora. não posso guardá-los. então, e depois eu cheguei...',
 'boa tarde amanina. é toda nervosa que vem de lá de baixo. homem, eu já encontrei. contra o quê? encontrei lisa. então eu vou passar aqui às relações públicas e diz, está bem? está lá, está lá. rolações públicas, boa tarde. olha, menina, eu já encontrei. encontrou o quê? a elisa, que ela mora aqui embaixo. ah, ok. vocês todas as noites perguntam. uma vizinh

{'WER_CALC': [0.5228758169934641, 0.45045045045045046, 0.423728813559322, 0.39215686274509803, 0.5705128205128205, 0.3916083916083916, 0.5980392156862745, 0.526829268292683, 0.425531914893617, 0.5119047619047619, 0.43125, 0.45045045045045046, 0.39520958083832336, 0.3778801843317972], 1: 0.5228758169934641, 2: 0.45045045045045046, 3: 0.423728813559322, 4: 0.39215686274509803, 5: 0.5705128205128205, 6: 0.3916083916083916, 7: 0.5980392156862745, 8: 0.526829268292683, 9: 0.425531914893617, 10: 0.5119047619047619, 11: 0.43125, 12: 0.45045045045045046, 13: 0.39520958083832336, 14: 0.3778801843317972}
{'CER_CALC': [0.289875173370319, 0.18397085610200364, 0.18197573656845753, 0.19698492462311556, 0.40620592383638926, 0.14725568942436412, 0.3292929292929293, 0.30173292558613657, 0.2111368909512761, 0.375, 0.197352587244284, 0.22804054054054054, 0.19416243654822335, 0.1657754010695187], 1: 0.289875173370319, 2: 0.18397085610200364, 3: 0.18197573656845753, 4: 0.19698492462311556, 5: 0.40620592383

<p align = "center">
    <img src = "evals\Latex Table Sistema V9.png">
</p>